# DEAI-opdrachten Classificatie

Dit notebook bevat de implementatie van twee classificatiemodellen voor de Ames Housing dataset:
- Binaire classificatie: Voorspellen of een huis een garage heeft
- Multi-class classificatie: Voorspellen van het kwaliteitsniveau van het huis

## 1. Dataset inladen

In deze sectie lezen we de AmesHousing dataset in vanuit het Excel-bestand.

In [ ]:
import pandas as pd

df = pd.read_excel('AmesHousing.xlsx', sheet_name='AmesHousing')

print(df.head())

print(df.columns.tolist())

   ID  SalePrice Garage  Overall Qual  Gr Liv Area  Total Bsmt SF  Lot Area  \
0   1     215000    yes             6         1656         1080.0     31770   
1   2     105000    yes             5          896          882.0     11622   
2   3     172000    yes             6         1329         1329.0     14267   
3   4     244000    yes             7         2110         2110.0     11160   
4   5     189900    yes             5         1629          928.0     13830   

   Year Built  Full Bath  Bedroom AbvGr Neighborhood House Style  
0        1960          1              3        NAmes      1Story  
1        1961          1              2        NAmes      1Story  
2        1958          1              3        NAmes      1Story  
3        1968          2              3        NAmes      1Story  
4        1997          2              3      Gilbert      2Story  
['ID', 'SalePrice', 'Garage', 'Overall Qual', 'Gr Liv Area', 'Total Bsmt SF', 'Lot Area', 'Year Built', 'Full Bath', 'Bedro

## 2. Datavorbereid binaire classificatie (Garage aanwezig)



In [ ]:
features_binary = ['Lot Area', 'Year Built', 'Neighborhood']
target_binary = 'Garage'

df_binary = df[features_binary + [target_binary]].copy()

df_binary = pd.get_dummies(df_binary, columns=['Neighborhood'], drop_first=True)

X_binary = df_binary.drop(target_binary, axis=1)
y_binary = df_binary[target_binary]

from sklearn.model_selection import train_test_split
X_train_binary, X_test_binary, y_train_binary, y_test_binary = train_test_split(X_binary, y_binary, test_size=0.2, random_state=42)

print("Binary classification data prepared.")
print("X_train shape:", X_train_binary.shape)
print("X_test shape:", X_test_binary.shape)

Binary classification data prepared.
X_train shape: (2344, 29)
X_test shape: (586, 29)


## 3. Datavorbereid multi-class classificatie (Kwaliteitsniveau)


In [ ]:
features_multi = ['Gr Liv Area', 'Year Built', 'House Style']
target_multi = 'Overall Qual'

df_multi = df[features_multi + [target_multi]].copy()

df_multi = pd.get_dummies(df_multi, columns=['House Style'], drop_first=True)

X_multi = df_multi.drop(target_multi, axis=1)
y_multi = df_multi[target_multi]

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

print("Multi-class classification data prepared.")
print("X_train shape:", X_train_multi.shape)
print("X_test shape:", X_test_multi.shape)

Multi-class classification data prepared.
X_train shape: (2344, 9)
X_test shape: (586, 9)


## 4. Decision Tree trainen - Binaire classificatie

Onderzoek hyperparameters met help().

Gekozen hyperparameters: max_depth=5, min_samples_split=10

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_binary = DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42)
dt_binary.fit(X_train_binary, y_train_binary)

print("Binary classification model trained.")

Help on function __init__ in module sklearn.tree._classes:

__init__(
    self,
    *,
    criterion='gini',
    splitter='best',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    random_state=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    class_weight=None,
    ccp_alpha=0.0,
    monotonic_cst=None
)
    Initialize self.  See help(type(self)) for accurate signature.

Binary classification model trained.


## 5. Binair classificatiemodel evalueren

Bereken accuracy, precision, recall, F1-score.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

y_pred_binary = dt_binary.predict(X_test_binary)

accuracy = accuracy_score(y_test_binary, y_pred_binary)
precision = precision_score(y_test_binary, y_pred_binary, pos_label='yes')
recall = recall_score(y_test_binary, y_pred_binary, pos_label='yes')
f1 = f1_score(y_test_binary, y_pred_binary, pos_label='yes')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_binary, y_pred_binary))

print("\nVoorbeelden van voorspellingen:")
print("Echte waarde vs Voorspelling:")
for i in range(5):
    print(f"Huis {i+1}: Echt={y_test_binary.iloc[i]}, Voorspeld={y_pred_binary[i]}")

Accuracy: 0.9300
Precision: 0.9379
Recall: 0.9909
F1 Score: 0.9637

Classification Report:
              precision    recall  f1-score   support

          no       0.17      0.03      0.05        37
         yes       0.94      0.99      0.96       549

    accuracy                           0.93       586
   macro avg       0.55      0.51      0.51       586
weighted avg       0.89      0.93      0.91       586



## 6. Decision Tree trainen - Multi-class classificatie

Gekozen hyperparameters: max_depth=6, min_samples_leaf=5

In [ ]:
dt_multi = DecisionTreeClassifier(max_depth=6, min_samples_leaf=5, random_state=42)
dt_multi.fit(X_train_multi, y_train_multi)

print("Multi-class classification model trained.")

Multi-class classification model trained.


## 7. Multi-class classificatiemodel evalueren

Bereken accuracy, precision, recall, F1-score per class.

In [ ]:
y_pred_multi = dt_multi.predict(X_test_multi)

accuracy_multi = accuracy_score(y_test_multi, y_pred_multi)
precision_multi = precision_score(y_test_multi, y_pred_multi, average='weighted')
recall_multi = recall_score(y_test_multi, y_pred_multi, average='weighted')
f1_multi = f1_score(y_test_multi, y_pred_multi, average='weighted')

print(f"Accuracy: {accuracy_multi:.4f}")
print(f"Precision (weighted): {precision_multi:.4f}")
print(f"Recall (weighted): {recall_multi:.4f}")
print(f"F1 Score (weighted): {f1_multi:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_multi, y_pred_multi))

Accuracy: 0.5000
Precision (weighted): 0.4754
Recall (weighted): 0.5000
F1 Score (weighted): 0.4775

Classification Report:
              precision    recall  f1-score   support

           2       0.67      0.67      0.67         3
           3       0.00      0.00      0.00        10
           4       0.33      0.10      0.16        49
           5       0.51      0.73      0.60       150
           6       0.37      0.38      0.38       119
           7       0.61      0.58      0.59       137
           8       0.54      0.51      0.52        83
           9       0.42      0.42      0.42        26
          10       0.00      0.00      0.00         9

    accuracy                           0.50       586
   macro avg       0.38      0.38      0.37       586
weighted avg       0.48      0.50      0.48       586



c:\Users\raymo\Documents\DEAI_portfolio\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\raymo\Documents\DEAI_portfolio\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\raymo\Documents\DEAI_portfolio\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

## 8. Experimenten - Binaire classificatie

Hier experimenten met verschillende features en hyperparameters om de prestaties te verbeteren.

In [ ]:
features_binary_exp1 = ['Lot Area', 'Year Built', 'Neighborhood', 'Gr Liv Area', 'Total Bsmt SF']
df_binary_exp1 = df[features_binary_exp1 + [target_binary]].copy()
df_binary_exp1 = pd.get_dummies(df_binary_exp1, columns=['Neighborhood'], drop_first=True)
X_binary_exp1 = df_binary_exp1.drop(target_binary, axis=1)
y_binary_exp1 = df_binary_exp1[target_binary]
X_train_exp1, X_test_exp1, y_train_exp1, y_test_exp1 = train_test_split(X_binary_exp1, y_binary_exp1, test_size=0.2, random_state=42)

dt_exp1 = DecisionTreeClassifier(max_depth=7, min_samples_split=5, random_state=42)
dt_exp1.fit(X_train_exp1, y_train_exp1)
y_pred_exp1 = dt_exp1.predict(X_test_exp1)
accuracy_exp1 = accuracy_score(y_test_exp1, y_pred_exp1)
print(f"Experiment 1 Accuracy: {accuracy_exp1:.4f}")

Experiment 1 Accuracy: 0.9300


## 9. Experimenten - Multi-class classificatie

Hier experimenten met verschillende features en hyperparameters om de prestaties te verbeteren.

In [ ]:
features_multi_exp1 = ['Gr Liv Area', 'Year Built', 'House Style', 'Lot Area', 'Full Bath']
df_multi_exp1 = df[features_multi_exp1 + [target_multi]].copy()
df_multi_exp1 = pd.get_dummies(df_multi_exp1, columns=['House Style'], drop_first=True)
X_multi_exp1 = df_multi_exp1.drop(target_multi, axis=1)
y_multi_exp1 = df_multi_exp1[target_multi]
X_train_exp1, X_test_exp1, y_train_exp1, y_test_exp1 = train_test_split(X_multi_exp1, y_multi_exp1, test_size=0.2, random_state=42)

dt_exp1_multi = DecisionTreeClassifier(max_depth=8, min_samples_leaf=3, criterion='entropy', random_state=42)
dt_exp1_multi.fit(X_train_exp1, y_train_exp1)
y_pred_exp1_multi = dt_exp1_multi.predict(X_test_exp1)
accuracy_exp1_multi = accuracy_score(y_test_exp1, y_pred_exp1_multi)
print(f"Experiment 1 Multi Accuracy: {accuracy_exp1_multi:.4f}")

Experiment 1 Multi Accuracy: 0.4949


## Logboek

### Model 1: Binaire classificatie
- Target: Garage (yes/no)
- Features: Lot Area, Year Built, Neighborhood
- Hyperparameters: max_depth=5, min_samples_split=10
- Evaluatie: accuracy 0.9300, precision 0.9379, recall 0.9909, F1 0.9637
- Experiment 1: extra features met max_depth=7, min_samples_split=5 → accuracy 0.9300, geen verbetering

### Model 2: Multi-class classificatie
- Target: Overall Qual (1-10)
- Features: Gr Liv Area, Year Built, House Style
- Hyperparameters: max_depth=6, min_samples_leaf=5
- Evaluatie: accuracy 0.5000, weighted precision 0.4754, recall 0.5000, F1 0.4775
- Experiment 1: extra features met criterion='entropy' → accuracy 0.4949, lichte daling

Dit logboek bevat de belangrijkste keuzes, resultaten en experimenten voor beide classificatiemodellen.